# The Chi-Square Test
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** the difference between a chi-square goodness-of-fit test and a chi-square test of independence
- **Calculate** a chi-square statistic by comparing observed and expected frequencies
- **Interpret** what it means for two categorical variables to be statistically independent

> **Tip:** Start by looking at the **observed vs expected bars** before computing anything, try to guess which categories deviate most from expectation, then verify with the χ² statistic.

---
## How we got here

In *17–20* every test we ran compared **means**, the t-test, the z-test, and hypothesis testing all focused on numeric outcomes. But many real-world variables are categorical: browser type, product preference, political affiliation, disease diagnosis. The chi-square test is how we apply hypothesis testing logic to **counts and frequencies** rather than means.

---
## Why this matters for data science

Chi-square tests appear throughout data science whenever you work with categorical data. Feature selection for categorical variables often uses chi-square to identify which categories are associated with a target label. A/B tests on click rates, survey response comparisons, and checks for whether a classifier's errors are independent of demographic groups all use chi-square logic. It is one of the most versatile tests in the practitioner's toolkit.

---
## Try it yourself

In [2]:
from plotly.subplots import make_subplots
from tkh_utils import make_int_slider, make_output, display_widget, register_observer

_CAT_LABELS = ["Strongly Agree", "Agree", "Disagree", "Strongly Disagree"]

out = make_output()
s1 = make_int_slider(1, 100, 1, 40, "Strongly Agree:")
s2 = make_int_slider(1, 100, 1, 30, "Agree:")
s3 = make_int_slider(1, 100, 1, 20, "Disagree:")
s4 = make_int_slider(1, 100, 1, 10, "Strongly Disagree:")

def render(o1, o2, o3, o4):
    observed  = np.array([o1, o2, o3, o4], dtype=float)
    n_total   = observed.sum()
    expected  = np.full(4, n_total / 4)
    chi2_stat, p_val = stats.chisquare(observed)
    df_chi = 3

    x_chi   = np.linspace(0.001, max(chi2_stat * 2.5, 15), 400)
    y_chi   = stats.chi2.pdf(x_chi, df_chi)
    x_shade = x_chi[x_chi >= chi2_stat]
    y_shade = stats.chi2.pdf(x_shade, df_chi)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Observed vs Expected Counts", "Chi-Square Distribution"],
    )
    fig.add_trace(go.Bar(
        x=_CAT_LABELS, y=observed,
        name="Observed", marker_color=PALETTE["primary"], opacity=0.85,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=_CAT_LABELS, y=expected,
        name="Expected (uniform)", marker_color=PALETTE["muted"], opacity=0.75,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x_chi, y=y_chi,
        mode="lines", line=dict(color=PALETTE["primary"], width=2.5),
        name=f"χ²(df = {df_chi})",
    ), row=1, col=2)
    if len(x_shade) > 1:
        fig.add_trace(go.Scatter(
            x=np.concatenate([[x_shade[0]], x_shade, [x_shade[-1]]]),
            y=np.concatenate([[0], y_shade, [0]]),
            fill="toself", fillcolor="rgba(247,103,7,0.35)",
            line=dict(width=0),
            name=f"p-value = {p_val:.4f}",
        ), row=1, col=2)
    fig.add_vline(
        x=chi2_stat,
        line_color=PALETTE["secondary"], line_width=2, line_dash="dash",
        annotation_text=f"χ² = {chi2_stat:.2f}",
        row=1, col=2,
    )
    layout = base_layout(
        title=f"χ² = {chi2_stat:.2f}, df = {df_chi}, p = {p_val:.4f}",
        xaxis_title="",
        yaxis_title="Count",
    )
    fig.update_layout(layout)
    fig.update_layout(barmode="group", showlegend=True)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

register_observer([s1, s2, s3, s4], lambda change: render(s1.value, s2.value, s3.value, s4.value))
display_widget([s1, s2, s3, s4], out)
render(s1.value, s2.value, s3.value, s4.value)


---
## What's happening?

The chi-square test compares what we **observed** in our data to what we would **expect** if the null hypothesis were true.

| Term | Meaning |
|------|---------|
| O | Observed count in a category |
| E | Expected count under H₀ |
| χ² | Chi-square statistic: total standardized deviation |
| df | Degrees of freedom: depends on number of categories and groups |

$$\chi^2 = \sum_{i} \frac{(O_i - E_i)^2}{E_i}$$

**Goodness-of-fit test:** Are the observed category frequencies consistent with a specified distribution?

**Test of independence:** Are two categorical variables associated, or are they independent of each other?

$$df_{\text{goodness-of-fit}} = k - 1 \qquad df_{\text{independence}} = (r-1)(c-1)$$

### The intuition behind the formula

Each term (O − E)²/E measures how surprising one category's count is relative to expectation. Large deviations get squared (always positive) and normalized by E (so rare categories don't dominate unfairly). Summing across all categories gives the total "surprise." A large χ² means the data looks very different from what H₀ predicts.

---
## Real-world example: Browser preference and subscription rate

A streaming service wants to know whether the browser a user uses is associated with whether they subscribe. This is a chi-square test of independence: are browser type (Chrome, Firefox, Safari, Edge) and subscription status (Yes/No) independent?

The chart shows observed subscription counts by browser alongside the expected counts if browser and subscription were truly independent. Notice:

- **Notice:** Safari users subscribe at a noticeably higher rate than expected, the observed count in the "Safari, Yes" cell is meaningfully above expected
- **Notice:** Edge users subscribe at a lower rate than expected, but the cell count is also small, which reduces its contribution to χ²
- **Notice:** The chi-square statistic captures all four browsers' deviations simultaneously, no single cell tells the whole story

> **Discussion question:** A marketing team sees the Safari finding and wants to target Safari users with premium ads. Before they act, what alternative explanations should they rule out? (Hint: think about who uses Safari versus Chrome in terms of demographics and device type.)

In [3]:
# ── Chi-square test of independence: browser vs subscription ─────────────────
# Simulated data from a realistic web analytics scenario
browsers  = ["Chrome", "Firefox", "Safari", "Edge"]
subscribed    = np.array([320, 145, 210, 78])    # observed subscribers per browser
not_subscribed = np.array([980, 410, 490, 290])  # observed non-subscribers per browser

observed = np.array([subscribed, not_subscribed])
chi2, p_val, df, expected = stats.chi2_contingency(observed)

fig = go.Figure()
for row_idx, (label, color) in enumerate(
    [("Subscribed", PALETTE["accent"]), ("Not Subscribed", PALETTE["muted"])]
):
    obs_row = observed[row_idx]
    exp_row = expected[row_idx]
    for i, browser in enumerate(browsers):
        fig.add_trace(go.Bar(
            x=[browser],
            y=[obs_row[i]],
            marker_color=color,
            name=label if i == 0 else "",
            showlegend=(i == 0),
            offsetgroup=row_idx,
        ))
        # Expected as a small marker
        fig.add_trace(go.Scatter(
            x=[browser], y=[exp_row[i]],
            mode="markers",
            marker=dict(symbol="line-ew", size=20, color="black",
                        line=dict(color="black", width=2)),
            name="Expected" if (row_idx == 0 and i == 0) else "",
            showlegend=(row_idx == 0 and i == 0),
        ))

layout = base_layout(
    title=f"Browser vs Subscription — Chi-Square Test of Independence<br>"
          f"χ²({df}) = {chi2:.2f},  p = {p_val:.4f}",
    xaxis_title="Browser",
    yaxis_title="Count",
)
layout.update(barmode="group")
fig.update_layout(layout)
fig.show()

### Chi-square test variants and their uses

| Test variant | What it tests | Example |
|-------------|--------------|---------|
| Goodness of fit | Does one categorical variable match an expected distribution? | Is a die fair? (each face should appear 1/6 of the time) |
| Test of independence | Are two categorical variables associated? | Is browser choice independent of subscription? |
| Test of homogeneity | Do two or more groups have the same distribution of a categorical variable? | Do men and women have the same product preference? |
| McNemar's test | Paired before/after categorical comparison | Did opinions change after an intervention? |
| Fisher's exact test | Independence with very small expected counts | Rare-event clinical trial with small cells |

---
## Key takeaway

> **The chi-square statistic sums (Observed − Expected)²/Expected across all cells; a large value means the categorical pattern in your data is too far from expectation to be explained by chance.**

---
*You have reached the end of this statistics series. The concepts here, from basic probability through chi-square tests, are the foundation for every machine learning model, every evaluation metric, and every data-driven decision you will encounter in the course ahead.*